# BorrowBox — Current Implementation State

**Purpose.** This notebook is an implementation-reality bridge. The other notebooks in `docs/` (`BORROWBOX_PRODUCT_SPEC`, `BORROWBOX_TRANSACTION_SPEC`, `BORROWBOX_ROADMAP`, `BORROWBOX_DECISIONS`, `BORROWBOX_ARCHITECTURE`, `BORROWBOX_V2_*`) describe intent. This notebook describes what is actually in the repository at the current HEAD, and where the two diverge. It does not restate specifications and it does not reproduce source code.

## Status labels used throughout

| Label | Meaning |
|---|---|
| **Implemented** | Code exists in the repository at this HEAD. |
| **Verified** | Executed in this working tree at this HEAD and observed to pass. |
| **Test Gap** | Implemented, but not covered by a passing test, or coverage unverified. |
| **Deferred** | Intentionally not built yet; recorded in the roadmap/ADRs as later-stage. |
| **Not Implemented** | Named in a specification but absent from the code, with no deferral decision recorded. |

Only **Verified** claims come from a run performed for this notebook. Everything else is read from the code and the schema.

## Current Git checkpoint

| Field | Value |
|---|---|
| Branch | `main` |
| HEAD | `5d547b5` — `fix: harden nullable date rendering in dashboard` |
| Working tree | clean (no modified, staged, or untracked files) |

The recent slice history on this branch runs: V2.5.1 lender evidence and condition metadata → a P1 bug-fix slice → a Phase 1 slice adding controller and cross-community isolation tests → the nullable-date rendering fix now at HEAD.

The repository is therefore at a **post-feature hardening point**, not inside an open feature slice. Every statement below describes this commit, not an intended design.

## Technology stack

| Layer | Technology in use |
|---|---|
| Runtime | Java 21, Spring Boot 3.5.1, Maven |
| Persistence | Spring Data JPA / Hibernate, MySQL 8 (InnoDB), HikariCP, dynamic JPA Specifications |
| Security | Spring Security, stateless JWT in HttpOnly cookies (jjwt 0.11.5), BCrypt |
| API documentation | springdoc-openapi 2.8.9 (Swagger UI) |
| Frontend | React 18.2, Vite 5, react-router-dom 7, hand-written CSS design system |
| End-to-end testing | Cypress 15.21 |
| Unit / integration testing | JUnit 5 via `spring-boot-starter-test`, Mockito, MockMvc, real-MySQL integration tests |
| Packaging | Multi-stage Docker builds, Docker Compose, Nginx reverse proxy, health-gated startup |

Schema management is deliberately **not** Hibernate-generated. `backend/src/main/resources/schema.sql` is version-controlled and applied with `spring.sql.init.mode=always`, while `ddl-auto=validate` makes Hibernate assert that the entity mappings match that checked-in file (ADR-018). There is no migration tool, so every schema change is an edit to `schema.sql` plus an entity change.

## Domain model

**Implemented.** Entities live in `backend/src/main/java/com/borrowbox/entity`.

Ownership is global and visibility is community-scoped (ADR-004): a user owns an `Asset`, an `Asset` materialises N `AssetUnit` rows as the real physical pool, and a `CommunityListing` selectively exposes that asset to one community. A `Transaction` is a single negotiation-and-loan record binding a listing, a borrower, a lender, a community, and at most one reserved `AssetUnit`.

Key enums exactly as implemented:

| Enum | Values |
|---|---|
| `TransactionStatus` | PENDING, APPROVED, REJECTED, COUNTER_OFFERED, CANCELLED, AWAITING_HANDOVER, ACTIVE, RETURN_INITIATED, RETURN_REPORTED, COMPLETED, HANDOVER_DISPUTED, RETURN_DISPUTED |
| `AssetUnitStatus` | AVAILABLE, RESERVED, BORROWED, NOT_AVAILABLE, LOST, DAMAGED, ARCHIVED |
| `EvidenceType` | LENDER_PRE_LENDING, LENDER_HANDOVER, BORROWER_PRE_RETURN, BORROWER_RETURN_HANDOVER |
| `TransactionEventType` | 17 semantic types across request, handover, extension, return, and waitlist promotion |
| `CommunityRuleType` | MEMBERSHIP_CONTEXT_FIELDS, MAX_ACTIVE_MEMBERS, ADMISSION_NOTE, OVERDUE_GRACE_PERIOD |
| Supporting | `CommunityType`, `CommunityStatus`, `CommunityAdmissionMode`, `AdmissionDecision`, `MembershipRole`, `MembershipStatus`, `ListingStatus`, `AssetStatus`, `FlagStatus`, `WaitlistStatus` |

Two structural notes. `AssetUnitStatus` values `LOST`, `DAMAGED`, `ARCHIVED`, and `NOT_AVAILABLE` exist in the enum but no flow in the codebase ever produces them — see Deferred work. And there is no separate Loan, Extension, or Dispute entity: all of that state lives as columns on the single `transactions` row, which is the main reason the transaction entity is unusually wide.

## Transaction state machine

**Implemented.** All transitions are enforced server-side in `TransactionService` against a pessimistic-lock read of the row. There is no client-driven state and no partial-patch escape hatch.

```text
Negotiation
  PENDING ── approve (lender) ──────────────> APPROVED
          ── reject (lender) ───────────────> REJECTED          [terminal]
          ── counter-offer (lender) ────────> COUNTER_OFFERED
          ── cancel (borrower) ─────────────> CANCELLED         [terminal]

  COUNTER_OFFERED ── accept-counter (borrower) > APPROVED
                 ── cancel (borrower) ──────> CANCELLED         [terminal]

Handover
  APPROVED ── stage-handover (either) ──────> AWAITING_HANDOVER
          ── cancel (either) ──────────────> CANCELLED
  AWAITING_HANDOVER ── confirm-handover ────> ACTIVE
                ── cancel (either) ────────> CANCELLED
                ── dispute-handover ────────> HANDOVER_DISPUTED  [terminal]

  ACTIVE ── confirm-receipt (borrower) ─────> ACTIVE   (self-transition,
                                              stamps borrowerConfirmedAt)

Return
  ACTIVE ── initiate-return (borrower) ─────> RETURN_INITIATED
  RETURN_INITIATED ── report-handback ──────> RETURN_REPORTED
  RETURN_REPORTED ── confirm-return ────────> COMPLETED        [terminal]
  RETURN_REPORTED ── dispute-return ────────> RETURN_DISPUTED  [terminal]
```

Facts worth holding onto: `APPROVED` is **not** terminal — it advances to `AWAITING_HANDOVER` and can still be cancelled by either party. `DUE_SOON` and `OVERDUE` are derived read-time conditions and are deliberately not persisted states. `RESERVED` is an `AssetUnitStatus`, never a transaction state.

## Handover lifecycle

**Implemented.**

1. Either participant calls `stage-handover` (`APPROVED → AWAITING_HANDOVER`). The reservation is kept and the unit stays `RESERVED`.
2. While `AWAITING_HANDOVER`, the **lender** uploads borrow-side evidence — `LENDER_PRE_LENDING` and `LENDER_HANDOVER` — optionally with a condition note and a 1–5 condition rating.
3. The lender calls `confirm-handover`. This is rejected unless at least one `LENDER_HANDOVER` evidence row already exists; that evidence gate was added in V2.5.1 and is enforced by querying the evidence repository rather than trusting the client.
4. On success the transaction becomes `ACTIVE`, the unit flips `RESERVED → BORROWED`, and the authoritative loan clock is stamped from the server clock: `startedAt = now`, `dueAt = startedAt + agreedDurationDays`, `originalDueAt = dueAt`.

A 30-minute confirmation window (`HANDOVER_WINDOW_MINUTES`) opens at handover and is derived from the server-stamped `startedAt`, never from client input. Inside it the borrower may either confirm receipt (stamps `borrowerConfirmedAt`; the transaction stays `ACTIVE`) or dispute non-receipt. Outside the window both actions are rejected, and a second confirmation is rejected outright.

The system takes no position on fault at this point. The borrower has exactly two choices and confirming is the non-committal one; a dispute is the borrower declining to accept responsibility, not a finding against the lender.

## Return lifecycle

**Implemented.**

1. The borrower calls `initiate-return` (`ACTIVE → RETURN_INITIATED`). This is rejected while any extension negotiation is still pending. The unit stays `BORROWED` — starting a return does not close the loan.
2. While `RETURN_INITIATED`, the **borrower** uploads both return-side evidence moments: `BORROWER_PRE_RETURN` and `BORROWER_RETURN_HANDOVER`.
3. The borrower calls `report-handback` (`RETURN_INITIATED → RETURN_REPORTED`). This is rejected unless both return-side photos exist. The unit still stays `BORROWED`.
4. The **lender** then closes the loop from `RETURN_REPORTED` only:
   - `confirm-return` → `COMPLETED`, `completedAt` stamped, unit `BORROWED → AVAILABLE`, reservation handle cleared, reputation rows appended, waitlist promotion attempted.
   - `dispute-return` → `RETURN_DISPUTED`.

The separation of `report-handback` and `confirm-return` is the substantive design point: a lender cannot complete a loan the borrower has not reported as physically handed back, so the item cannot be re-lent on the lender's word alone. `RETURN_INITIATED` and `RETURN_REPORTED` are both states in which the unit is deliberately still `BORROWED`.

## Disputes

**Implemented** for recording and terminal freeze. **Deferred** for resolution.

Two dispute kinds exist, and they differ deliberately in what happens to the physical unit:

| | Handover dispute | Return dispute |
|---|---|---|
| Trigger | borrower, inside the 30-minute window, from `ACTIVE` | lender, from `RETURN_REPORTED` |
| Resulting state | `HANDOVER_DISPUTED` (terminal) | `RETURN_DISPUTED` (terminal) |
| AssetUnit | released `BORROWED → AVAILABLE`, reservation cleared | **stays `BORROWED`**, reservation handle retained |
| Waitlist promotion | attempted, since a unit became free | not attempted, since the unit is not re-lendable |
| Audit trail | SYSTEM timeline event | `returnDisputedAt` / `returnDisputedBy`, SYSTEM event, one borrower reputation row with `successful=false`, `on_time=false` |
| Evidence | preserved, rows are immutable | preserved, rows are immutable |

Both states are read-only: there are no forward transitions in either direction, and the code contains no reopen path.

The asymmetry is the interesting part of the design. If the borrower disputes non-receipt, nobody has the item, so the unit is released immediately and the next waiter is promoted. If the lender disputes the return, the item is in an unknown physical location, so the unit is frozen and blocked from re-lending — the cost of the ambiguity falls on availability rather than on either party.

A community manager can observe these incidents through community health, flags, and the manager dashboard, but **cannot** resolve, reopen, or adjudicate a disputed transaction. No code path grants that power.

**Not Implemented:** the specification's other dispute examples — pre-existing damage, wrong item, missing component. A dispute can currently only be raised as non-receipt or not-received.

## Extensions

**Implemented.** An extension is a negotiation that never changes the transaction state — the loan stays `ACTIVE` throughout, and no new `TransactionStatus` exists for it.

- The borrower requests a new absolute `dueAt` while `ACTIVE`, including while the loan is currently overdue.
- The lender may accept, reject, or counter; the borrower must then explicitly accept or reject a counter. There is one lender round only — no re-countering.
- Exactly one negotiation may be pending at a time. The pending request is held in four nullable columns on the transaction (`extensionRequestedDueAt`, `extensionOfferedDueAt`, `extensionNote`, `extensionRequestedAt`) and exposed to clients as derived booleans, never as a state.
- Validation applies to both requests and counters: the new date must be strictly after the current `dueAt`, strictly after the server clock, and no more than `EXTENSION_MAX_DAYS` (30) after the current `dueAt`.
- On acceptance `dueAt` moves. `originalDueAt` and `agreedDurationDays` are never rewritten, so the originally agreed term survives any number of extensions. Accepting an extension while overdue immediately re-derives `OVERDUE = false`.
- On resolution the four negotiation columns are cleared. The durable history is the SYSTEM conversation timeline — there is no dedicated extension history table.
- `initiate-return` is rejected while a negotiation is pending, and a handover dispute clears the pending fields before recording `HANDOVER_DISPUTED`, so no terminal row carries orphaned extension data.
- Extensions never touch unit, reservation, or availability state.

## Overdue and grace-period logic

**Implemented** — the V2.2.4 accountability clock plus the V2.4.1 community grace period. There is no scheduler anywhere in the backend.

- `DUE_SOON` is derived on read: the state is `ACTIVE`, and the current time falls within 24 hours (`DUE_SOON_THRESHOLD_HOURS`) before `dueAt`.
- `OVERDUE` is derived on read: the state is `ACTIVE`, and the current time is past the effective deadline.
- The effective deadline is `dueAt + graceDays`, where `graceDays` comes from the transaction's community and its active `OVERDUE_GRACE_PERIOD` rule. The value is read from that rule's JSON `days` field and clamped to non-negative; a missing rule, a zero, or an unparseable value means zero grace, which preserves the pre-V2.4.1 behaviour exactly.
- The grace period is applied **only at evaluation time**. `startedAt`, `dueAt`, and `originalDueAt` are never rewritten to absorb it, so the authoritative loan clock stays identical regardless of which community is asking or when it is asked.
- Both conditions require state `ACTIVE`, so a loan that has moved to `RETURN_INITIATED` or `RETURN_REPORTED`, or reached any terminal state, never reports as overdue. A late item being returned does not accrue overdue status.

The practical consequence is that nothing happens on its own when a loan goes overdue. The condition is visible on read, but no reminder is sent, no flag is raised, and no escalation runs. That is **Not Implemented** and it is the main reason a grace period configured today has no effect beyond the derived boolean.

## Evidence and borrower notes

### Evidence

**Implemented.** All four moments from ADR-008 are enabled — the return side in V2.2.6, the borrow side in V2.5.1.

| Moment | Actor | Required state | Gate |
|---|---|---|---|
| `LENDER_PRE_LENDING` | lender | `AWAITING_HANDOVER` | — |
| `LENDER_HANDOVER` | lender | `AWAITING_HANDOVER` | at least one row required before `confirm-handover` |
| `BORROWER_PRE_RETURN` | borrower | `RETURN_INITIATED` | both rows required before `report-handback` |
| `BORROWER_RETURN_HANDOVER` | borrower | `RETURN_INITIATED` | both rows required before `report-handback` |

Authorisation is explicit by actor, evidence type, and state. Lenders may not upload return-side photos and borrowers may not upload borrow-side photos, and community managers get no bypass — the rule is participant-only, not moderator-inclusive, which is a stronger reading of ADR-011 than the specification's "participants/moderators" phrasing. Evidence rows are immutable: there is no update or delete path at all.

On storage: the binary is written by `EvidenceStorageService` under a UUID-only file reference that is never derived from user input, and only the reference, content type, size, and timestamps are persisted. Uploads must be images and are size-capped by `borrowbox.evidence.max-size-bytes` (default 5 MB). If the enclosing database transaction rolls back, a transaction synchronisation deletes the just-written file, so a failed insert leaves no orphaned binary on disk. Content is never served statically; it is reachable only through the authenticated, participant-checked content endpoint.

Condition metadata is per evidence row (`conditionNote`, `conditionRating`). An out-of-range rating is rejected rather than silently clamped. `AssetUnit.condition` exists in the schema but is unmanaged — condition is never written back to the unit, so there is currently no durable unit-level condition history.

### Borrower and participant notes

**Implemented.** Free text is carried at four points with fixed caps: the borrower's `borrower_note` on the request, the lender's `counter_note` on a counter-offer, `decision_note` on approve or reject, and `extension_note` during extension negotiation. Blank values are normalised to null rather than stored as empty strings.

Separately, `transaction_messages` provides a per-transaction conversation. `USER` rows carry an author and hold pickup-coordination and logistics messages; `SYSTEM` rows have a null author and are timeline entries written by the backend on each transition. The null `author_id` is the enforcement mechanism — SYSTEM rows cannot be created through the public message API. All message, event, and lifecycle timestamps are server-generated; client clocks are never trusted (ADR-006).

## Waitlist

**Implemented** (V2.2.7).

- The queue is **per Asset** and shared across every community that lists that asset, consistent with the shared-physical-inventory model. A borrower enters through one specific `CommunityListing`, and `listing_id` pins which community they queued in; `community_id` is derivable and deliberately not duplicated.
- Joining is permitted only when **zero** `AssetUnit` rows are `AVAILABLE`. The check takes the same pessimistic `AssetUnit` lock that request creation uses, so this is a serialised availability test rather than a count-then-insert race. A borrower is told to submit a normal request instead.
- `UNIQUE(asset_id, borrower_id)` is the database authority for one live position per borrower per asset, backed by a pre-check and a translated integrity violation.
- Queue order is `created_at` then `id`, both server-stamped. Position is **derived on read** as `countWaitingBefore + 1` and is never persisted, so it cannot go stale.
- Promotion is **synchronous**, inside the same database transaction that made a unit `AVAILABLE`. Reject, cancel, dispute-handover, confirm-return, and a voluntary leave all attempt it. `promoteForAsset` is `Propagation.MANDATORY` and loops until no unit or no waiter remains, so multiple freed units are drained in one call.
- The head entry is revalidated before promotion: listing still `LISTED`, asset not `ARCHIVED`, borrower not the owner, borrower still an active member of the queued community. An ineligible head is marked `LEFT` permanently and never revived, and the loop moves on to the next `WAITING` entry.
- A voluntary leave **hard-deletes** the `WAITING` row so the borrower may re-join, rather than marking it `LEFT`. Promotion then runs, so a unit freed in the meantime reaches the next waiter.
- A successful promotion creates an ordinary `PENDING` transaction, marks the entry `PROMOTED` with a server-stamped `promoted_at`, adds a SYSTEM timeline entry, and emits a `WAITLIST_PROMOTED` event with a null actor.
- `RETURN_DISPUTED` releases nothing and therefore triggers no promotion.
- There is no scheduler and no `@Async` promotion. If nothing frees a unit, nothing is promoted.

## Concurrency and locking model

**Implemented.** The backend and database are authoritative for both availability and ordering; client state and client clocks never decide an outcome (ADR-005, ADR-006).

| Concern | Mechanism |
|---|---|
| Picking a unit for a new request | `findFirstByAssetIdAndStatusForUpdate` — `PESSIMISTIC_WRITE` on the `AssetUnit` row |
| Mutating any transaction | `findByIdForUpdate` — `PESSIMISTIC_WRITE` on the `transactions` row |
| Exactly one winner per unit | pessimistic lock **plus** `UNIQUE(reserved_unit_id)` as a database backstop |
| Turning a lost race into a clean error | `saveAndFlush`, so a `UNIQUE` violation surfaces synchronously and becomes a `400 "No available unit"` rather than a deferred rollback-only failure |
| One waitlist position per borrower per asset | `UNIQUE(asset_id, borrower_id)` plus a pre-check |
| One delivery per recipient per event | `UNIQUE(event_id, recipient_id)` |
| No duplicate reputation rows | `UNIQUE(transaction_id, user_id, event_type)` plus an existence check before insert, so replaying a transition or a self-healing reconcile cannot duplicate a row |
| Lock ordering during promotion | always `AssetUnit` lock → waitlist head-row lock → transaction work; the inverse order is never introduced |

Further invariants held in code: an `AssetUnit` may participate in at most one non-terminated transaction; `REJECTED` and `CANCELLED` release the reservation immediately while `APPROVED` keeps it; and `AssetUnit` identifiers are never exposed in any response — `TransactionResponse` carries only a `unitReserved` boolean. Releasing a unit and promoting the next waiter happen in one transaction, so a freed unit is never briefly visible and unclaimed.

## Frontend structure

**Implemented.** React 18 + Vite SPA in `frontend/`, with no UI framework and no state-management library.

```text
src/
  pages/                 Landing, SignIn, SignUp
  pages/dashboard/       Dashboard, CommunityHome, Explore, Inventory, Requests,
                         Loans, Members, Rules, Flags, Profile, Settings
  components/ui/         Button, Input, Drawer, Avatar, Spinner, Toast, EventToast,
                         EmptyState, IconButton
  components/layout/     AppShell, PublicLayout, CommunityPageLayout, CommunityTabs,
                         CommunityPanel, CommunityHeader, UserBar, EventPanel,
                         EventEnvelope, SkipLink, Footer
  components/dashboard/  RequestDrawer, CounterOfferDrawer, ConversationDrawer,
                         ListAssetDrawer, ListingRowList, EventTimeline, HistoryList,
                         ReputationLedgerList, TrustSummary, RuleValueEditor
  components/routing/    LandingRedirect, CommunityRouteGuard
  contexts/              AuthContext, CommunityContext, EventContext, AppContext
  services/              request, waitlist, event, flag, reputation, rule, profile,
                         community, membership, listing, asset, category, auth
  hooks/                 useAsync, useCommunityRoute, useReducedMotion
  utils/                 apiClient.js
```

What this says about the product state: the request inbox, loans view, conversation drawer, event panel and envelope, flags page, rules page, and trust profile are all real screens rather than placeholders. Route protection is split between a generic auth guard and a community-scoped guard that also keeps manager-only surfaces manager-only, and there is a `SkipLink` and reduced-motion hook, so accessibility basics are in place.

The HEAD commit belongs to this layer: nullable transaction dates are now rendered defensively in the dashboard rather than assumed present.

## Database and schema overview

**Implemented.** MySQL 8 / InnoDB. The authoritative DDL is `backend/src/main/resources/schema.sql`, applied on every start and validated against the entity mappings. Sixteen tables:

| Group | Tables |
|---|---|
| Identity and community | `users`, `communities`, `memberships`, `community_rules` |
| Catalogue and inventory | `categories`, `assets`, `asset_units`, `community_listings` |
| Transaction core | `transactions` |
| Conversation and evidence | `transaction_messages`, `transaction_evidence` |
| Queueing | `waitlist_entries` |
| Events and delivery | `transaction_events`, `transaction_event_deliveries` |
| Trust | `reputation_events` |
| Moderation | `flags` |

Design points that matter when reading the code:

- `transactions.reserved_unit_id` is nullable and `UNIQUE`. MySQL permits many nulls in a unique column, so the constraint enforces "at most one open transaction holds a given physical unit" while still allowing any number of unreserved negotiations to coexist. It is a backstop behind the pessimistic lock, not the primary guard.
- `transactions` carries the entire negotiation and accountability history on one wide row: counter terms, agreed terms, decision fields, `started_at`, `due_at`, `original_due_at`, `borrower_confirmed_at`, the four extension columns, and `return_disputed_at` / `return_disputed_by`.
- `transaction_messages.kind` separates `USER` from `SYSTEM`, and the nullable `author_id` is what makes system timeline entries unforgeable through the public API.
- `reputation_events` is append-only, idempotent on `(transaction_id, user_id, event_type)`, and indexed by user, community, and transaction.
- `flags` is community-scoped with a nullable `transaction_id`, because a manual flag need not reference a transaction. It deliberately has **no** uniqueness on `(transaction_id, flag_type)`, so one transaction may legitimately carry several flags.
- Indexes cover the query paths that matter: transaction by lender/state, borrower/state, and listing/state; waitlist by asset/status/created_at; events by transaction/created_at; flags by community/type, community/status, transaction, and occurred_at.
- The file's header comment still reads "V2.2.8 schema baseline" even though V2.3.x, V2.4.x, and V2.5.x tables were appended by later slices — the schema grew incrementally rather than being re-baselined.

## Test coverage

### Backend

**Verified.** `cd backend && mvn test` at HEAD: **640/640 passing, 0 failures, 0 errors, 0 skipped**, `BUILD SUCCESS` against MySQL 8.0.46. The suite is not mocked-only — the integration tier runs against the real database.

- `service/` — unit tests with mocked collaborators, including 132 cases in `TransactionServiceTest` covering each transition, guard, and validation rule.
- `controller/` — MockMvc tests per controller.
- `integration/` — real-MySQL tests covering the transaction lifecycle, waitlist, trust profile, reputation ledger, flags and manager isolation, cross-community isolation, grace-period consistency, seed idempotence, schema validation, security rejection, and atomicity of asset and community creation.

### Frontend

**Verified** for the build only. `cd frontend && npm run build` succeeds and emits a production bundle with no build errors. There is no unit-test layer for frontend code.

### End-to-end

**Test Gap — not verified at this HEAD.**

- The suite is substantial: 21 spec files under `frontend/cypress/e2e/` declaring 148 `it(...)` cases, with no `it.skip` or `it.only` markers.
- A complete run was **not** performed for this notebook, so the suite is **not** claimed to be green. It requires the full stack serving on `localhost:3000` against a seeded database.
- The repository contains committed failure screenshots under `frontend/cypress/screenshots/` covering many specs — `waitlist`, `loan-lifecycle`, `extension`, `return-dispute`, `lender-evidence`, `events`, `role-access`, `manager-dashboard`, `membership-review`, `rules-page`, `profile`, `flag-management`, `completion`, and `conversation`. That is evidence of recorded failing runs; whether they still reproduce at HEAD is unverified.
- Documentation is stale here: `README.md` advertises "57 tests" across "6 spec files", and the Cypress badge reflects that figure. The spec files actually present are far more numerous than that.

## Completed development slices

**Implemented.** Reconstructed from the schema comments, entity javadoc, and branch history at HEAD.

| Slice | Delivered |
|---|---|
| V2.1.1 | Community + Membership foundation, roles/status/verification, `context_metadata` JSON, creator becomes manager |
| V2.1.2 | Admission via `MANAGER_APPROVAL` and `LOCATION_VERIFIED`, explicit non-continuous location check |
| V2.1.3 | Community rules foundation |
| V2.1.4 | Global asset ownership with atomic Asset + N AssetUnit creation |
| V2.1.5 | `CommunityListing` selective visibility with owner ACTIVE-membership authorisation |
| V2.1.6 | Community-scoped Explore, aggregate availability, cross-community shared inventory |
| V2.1.7 | Deterministic idempotent seed, local reset script, Docker verification |
| V2.2.1 | Structured requests, approve/reject/counter, race-safe reservation, request inboxes |
| V2.2.2 | Loan lifecycle `AWAITING_HANDOVER → ACTIVE → COMPLETED`, authoritative timestamps |
| V2.2.3 | Per-transaction conversation with SYSTEM timeline events |
| V2.2.4 | Accountability clock (`dueAt` / `originalDueAt`), 30-minute handover window, derived `DUE_SOON`/`OVERDUE` |
| V2.2.5 | Loan extensions: request, accept, reject, counter, accept-counter, counter-reject |
| V2.2.6 | Return disputes + transaction-scoped evidence with storage and content endpoint |
| V2.2.7 | Queueing / waitlist with synchronous promotion |
| V2.2.8 | Structured transaction events with per-recipient delivery and read/dismiss state |
| V2.3.1 | Derived trust profile, participant-only timeline, deterministic completed-transaction seed |
| V2.3.2 | Append-only reputation ledger (ADR-020, ADR-021) |
| V2.3.3 | Reliability metrics on the trust profile |
| V2.4.1 | Moderation flag foundation and `OVERDUE_GRACE_PERIOD` rule |
| V2.4.2 | Community health, membership moderation, manager dashboard |
| V2.4.3 | Rules UX and validation with typed editors instead of raw JSON |
| V2.5.1 | Lender borrow-side evidence + per-evidence condition metadata, with a handover-evidence gate |
| post-slice | P1 bug-fix slice; Phase 1 controller and cross-community isolation tests; nullable date rendering hardening |

The coherent result is a demonstrable vertical slice: request → approval → evidence-gated handover → accountable loan → extension negotiation → evidence-gated return → completion, with a trust ledger and moderation foundation behind it. Roadmap milestones v2.1 through v2.5 have each been delivered as a working slice.

## Known test gaps

**Test Gap.** Ordered by how much each one undermines confidence in the product.

1. **No verified full Cypress run at HEAD.** The E2E tier is the only layer that exercises the assembled product, and its status is currently unknown while committed failure screenshots sit in the repository. Until a full run is recorded, the project's own claim of a green E2E suite should not be relied on.
2. **Stale test-count documentation.** `README.md` and the Cypress badge state 57 tests across 6 spec files, against 21 spec files declaring 148 cases. Anyone triaging failures from the README is working from wrong numbers.
3. **Dispute resolution is untestable because it does not exist.** `HANDOVER_DISPUTED` and `RETURN_DISPUTED` are terminal, so there is nothing beyond the transition itself to cover.
4. **Time-dependent derived conditions are thin at the edges.** `DUE_SOON`, expiry of the 30-minute handover window, and overdue-with-grace all depend on wall-clock arithmetic. Service-level coverage exists in the backend suite, but an E2E assertion that the UI stops offering a window-closed action was not verified for this notebook.
5. **Lock-order discipline is documented, not enforced.** The `AssetUnit → waitlist → transaction` order in `promoteForAsset` is stated in comments as never to be inverted, and the request-creation race is covered, but a test that races two simultaneous promotions against one asset was not identified during this review.
6. **Evidence authorisation is covered at service and controller level; end-to-end proof of participant-only binary access was not verified.** The manager-bypass rule in particular deserves an explicit negative E2E test, because a regression there is a privacy failure under ADR-011 rather than a UI defect.
7. **Unreachable enum surface.** `AssetUnitStatus.LOST`, `DAMAGED`, `ARCHIVED`, and `NOT_AVAILABLE` are never produced by any flow, so nothing currently guards their eventual introduction.

## Deferred work

**Deferred** unless marked otherwise. These are named in the roadmap or specifications, or implied by the current design, and are intentionally not built.

| Area | Item | Note |
|---|---|---|
| Disputes | Resolution workflows for `HANDOVER_DISPUTED` and `RETURN_DISPUTED` | Both terminal today; a manager can see an incident but not adjudicate it |
| Disputes | Damage, wrong-item, missing-component, pre-existing-damage claim types | Only non-receipt and not-received can be raised |
| Handover | Representative handovers (account owner, physical participant, on behalf of, relationship) | Specified in `BORROWBOX_TRANSACTION_SPEC`; absent from the code |
| Automation | Scheduled reminders and due-soon/overdue notifications | No `@Scheduled` anywhere; all conditions are read-time derivations only |
| Notifications | Email, push, and webhook channels | Only in-app transaction events exist |
| Escalation | Configured escalation when a loan goes overdue | A grace period can be configured, but nothing acts on it |
| Expiry | `EXPIRED` state, reservation expiry, auto-release of stale reservations | Never in the enum; an abandoned reservation holds its unit indefinitely and never promotes the waitlist behind it |
| Trust | Badges, community recognition, reputation consequences of disputes | Reputation events are recorded but not yet consumed as consequences |
| Moderation | Flag escalation paths, a `FLAGGED` transaction state | Flags are community-scoped records, not transaction state |
| Inventory | `LOST` / `DAMAGED` / `ARCHIVED` / `NOT_AVAILABLE` unit lifecycle, and `AssetUnit.condition` | Enum values and a column exist; no flow writes them |
| AI | AI condition comparison, anomaly flags, blind tribunal, appeal, peer review | ADR-013 defers these until the evidence, transaction, and reputation foundations mature |
| Migration | V1 → V2 data migration | ADR-014: a fresh V2 database was a deliberate decision |
| Infrastructure | Community inventory endpoint | ADR-019: resolved as subsumed by `/me/inventory` plus community Explore |

The first three rows are the ones worth attention. A dispute that cannot be resolved is a permanently frozen loan in a product whose entire premise is trust between strangers.

## Specification vs implementation differences

**Reference.** Concrete divergences between the `docs/` specifications and the code at HEAD. The code is the reality this notebook documents; the specifications remain the record of intent.

### State machine naming and shape

| `BORROWBOX_TRANSACTION_SPEC` | Implemented | Note |
|---|---|---|
| `REQUESTED` | `PENDING` | renamed |
| `APPROVED → AWAITING_HANDOVER` | same | — |
| `HANDOVER_REPORTED` | *absent* | there is no report-then-confirm handover state; `confirm-handover` moves `AWAITING_HANDOVER → ACTIVE` in a single step |
| `ACTIVE` | same | — |
| `RETURN_INITIATED` | same | — |
| `RETURN_HANDOVER_REPORTED` | `RETURN_REPORTED` | renamed |
| `COMPLETED` | same | — |
| `DUE_SOON`, `OVERDUE` | derived, not persisted | the spec already frames these as derived, so implementation matches |
| exception `FLAGGED` | *absent* | flags live in their own community-scoped table, not as transaction state |
| exception `EXPIRED` | *absent* | never implemented |
| exceptions `CANCELLED`, `REJECTED`, `HANDOVER_DISPUTED`, `RETURN_DISPUTED` | all present | — |
| *not in spec* | `COUNTER_OFFERED` | added by the implementation, which the negotiation flow requires |

### Actor permissions

- The spec grants the borrower a neutral "report a problem" inside the handover window. The implementation offers exactly two borrower actions there: confirm receipt, or dispute non-receipt.
- The spec assigns the community manager flag and incident review plus moderation actions. The implementation gives managers flags, membership moderation, and a dashboard, but no power over a disputed transaction.
- The spec describes a System actor that sends reminders, generates notifications, and triggers configured escalations. The implementation has no scheduler and no such actor.
- The spec's representative-handover record (account owner, physical participant, on behalf of, relationship) is not implemented at all.

### Event-vs-state modelling

The spec says an extension is an event rather than a destructive update, and that history should retain the original due date, the request, the requested new date, the decision, and the final due date. The implementation does keep `originalDueAt` immutable and moves only `dueAt`, and it writes SYSTEM timeline entries — but the four negotiation columns are **cleared on resolution**, so the durable record lives in the conversation timeline rather than in a queryable extension history. This is the one place where the spec's audit requirement is met indirectly rather than directly.

### Repository documentation drift

- `PROJECT_STATUS.md` still describes the product as V2.2.1 (transaction negotiation) and states that `APPROVED` is "terminal in this slice". At HEAD, `APPROVED` is not terminal: it advances to `AWAITING_HANDOVER` and can still be cancelled by either party. That file predates V2.2.2 through V2.5.1.
- `README.md` carries a V2.2.2-era feature table, the "57 Cypress tests" figure, and a description in which `LoansPage` is a placeholder. Loans, events, flags, rules, and the trust profile are all implemented screens.
- `CHECKLIST.md` documents the V1 15-day build and says nothing about V2.

Taken together: the specifications are aspirational and the schema, entity javadoc, and this notebook are the accurate description of the product.

## Current development status

| Check | Result at HEAD |
|---|---|
| Backend `mvn test` | **640/640 passing**, 0 failures, 0 errors, 0 skipped — `BUILD SUCCESS` |
| Frontend `npm run build` | succeeds, production bundle emitted |
| Cypress E2E | **not run for this notebook; unverified — not claimed green** |
| Docker stack | not exercised for this notebook |
| Working tree | clean on `main` at `5d547b5` |

Interpretation:

- The backend is in a strong, verified position. The transaction engine, trust ledger, and moderation foundation are implemented and covered by a suite that exercises a real database rather than mocks.
- The project is **past feature slicing and into consolidation**. The last three commits are a P1 bug-fix slice, an added-tests slice, and a rendering fix — the signature of hardening rather than new capability.
- The largest source of uncertainty is not the backend. It is the **unknown state of the E2E tier and the documentation drift around it**. Two independent signals point at the same place: committed failure screenshots, and a README that understates the suite by a wide margin.
- The clearest structural gap between specification and product is **dispute resolution**. Two terminal states are reachable, both freeze the loan, and nothing can move them. For a trust-focused product this is the most consequential piece of unfinished business.
- `PROJECT_STATUS.md` should not be used to describe the current product. The schema, the entity javadoc, and this notebook are the reliable sources.

## Next candidate hardening work

**Proposals, not commitments.** Ordered by value relative to risk.

1. **Establish a verified E2E baseline.** Run the full Cypress suite against a freshly reset stack, triage what the committed screenshots point at, and either fix or explicitly quarantine each failure. Record the outcome so "E2E is green" becomes a checkable claim. Every other item on this list is weaker while the E2E state is unknown.
2. **Correct the documentation drift.** Bring `PROJECT_STATUS.md` up to the implemented state set and slice list, and fix the test and spec-file counts in `README.md`. Cheap, and it removes a recurring source of wrong assumptions about the project.
3. **Design dispute resolution before adding more dispute surface.** `HANDOVER_DISPUTED` and `RETURN_DISPUTED` are terminal. A manager-mediated resolution path — with an explicitly defined outcome for the AssetUnit and the reservation in each branch — is the highest-value missing capability, and the one place the current design is deliberately incomplete rather than merely absent.
4. **Close the concurrency-test gaps around promotion.** The documented lock order (`AssetUnit → waitlist → transaction`) is enforced by nothing but a comment. Add a test that races two promotions against a single asset, and one that races a voluntary `leave` against a freeing transition.
5. **Add the missing negative authorisation tests.** Managers must not bypass participant-only evidence access, and non-participants must not read evidence binaries, transaction timelines, or disputes. These are privacy invariants under ADR-011 and deserve explicit end-to-end proof rather than service-level assertions alone.
6. **Resolve the extension-audit question explicitly.** Either add a real extension history table, or record the timeline-event approach as an accepted ADR, so the spec's "an extension is an event" requirement is knowingly met indirectly rather than by omission.
7. **Decide what happens to stale reservations.** With no scheduler and no `EXPIRED` state, an abandoned `PENDING` or `AWAITING_HANDOVER` reservation holds a unit indefinitely and never promotes the waitlist behind it. In a queueing product this is a genuine availability defect, and it deserves a deliberate decision rather than continued silence.
8. **Give `AssetUnitStatus` a lifecycle or trim it.** `LOST`, `DAMAGED`, `ARCHIVED`, and `NOT_AVAILABLE` are unreachable today. Wire them to dispute resolution and manual moderation, or remove them so the enum stops implying behaviour the code does not have.